# MiraiTech: Анализ и извлечение шагов вокруг поворотов (Сессия #6546)

Этот Jupyter Notebook выполняет полный цикл биомеханического анализа челночного теста (Yo-Yo / Beep Test):
1. **Коррекция дрейфа гироскопа (Дрейф)**: Устранение накопленного ухода угла курса $XData$ (симметричный 31-секундный скользящий медианный фильтр).
2. **Нейросетевая детекция контактов (WalkBiLSTM)**: Точное покадровое определение моментов постановки и отрыва стопы (GCT).
3. **Детекция разворотов на 180°**: Нахождение 85 челночных виражей теста.
4. **Выделение шагов вокруг каждого разворота**: $Step_{-3}, Step_{-2}, Step_{-1}, Step_{\text{turn}}, Step_{+1}$.
5. **Расчет метрик**: Время контакта ($GCT$), время шага ($Step\;Time$), сила реакции опоры ($BW$ и $N$).
6. **Интерактивные графики и выгрузка в CSV**.

In [ ]:
import os
import io
import sys
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
import torch
import torch.nn as nn
import joblib

# Настройка путей к моделям и бэкенду
NOTEBOOK_DIR = Path(os.getcwd())
MODEL_DIR = NOTEBOOK_DIR / "models" / "walk_gc_bilstm"
if not MODEL_DIR.exists():
    MODEL_DIR = NOTEBOOK_DIR.parent / "MiraiTech-backend" / "app" / "services" / "calculators" / "models"

print(f"Используется каталог моделей: {MODEL_DIR}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Device: {device}")

## 1. Загрузка данных сессии

In [ ]:
# Укажите путь к CSV или Parquet файлу сессии
data_path = Path(r"C:\Users\Temirlan\Downloads\Сессия_#6546_labeled.csv")
if not data_path.exists():
    data_path = Path("Сессия_#6546_labeled.csv")

print(f"Загрузка: {data_path}...")
df_raw = pd.read_csv(data_path)

# Нормализация времени в миллисекунды
t_col = "Time" if "Time" in df_raw.columns else "time"
s_t = pd.to_numeric(df_raw[t_col].iloc[:300], errors="coerce").dropna()
if len(s_t) > 1 and np.median(np.diff(np.sort(s_t.values))) < 0.5:
    df_raw[t_col] = pd.to_numeric(df_raw[t_col], errors="coerce") * 1000.0

duration_s = (df_raw[t_col].max() - df_raw[t_col].min()) / 1000.0
print(f"Успешно загружено: {df_raw.shape[0]:,} строк, {df_raw.shape[1]} колонок | Длительность: {duration_s:.1f} сек ({duration_s/60:.1f} мин)")
df_raw.head()

## 2. Коррекция дрейфа гироскопа (Кнопка «Дрейф» для $XData$)

Гироскопы левой и правой стопы имеют температурный и аппаратный дрейф нуля. Поскольку оба датчика надеты на одного спортсмена, их реальный угол поворота совпадает, а разница $(Yaw_{L} - Yaw_{R})$ содержит только дрейф. Метод вычисляет 31-секундный скользящий тренд разницы и симметрично вычитает половину дрейфа из левой стопы и прибавляет к правой.

In [ ]:
def unwrap_angle_degrees(values: np.ndarray, threshold: float = 180.0) -> np.ndarray:
    if len(values) == 0:
        return values.astype(np.float64, copy=True)
    result = values.astype(np.float64, copy=True)
    offset = 0.0
    for i in range(1, len(values)):
        curr, prev = values[i], values[i - 1]
        if np.isnan(curr) or np.isnan(prev):
            continue
        diff = curr - prev
        if diff > threshold:
            offset -= 360.0
        elif diff < -threshold:
            offset += 360.0
        result[i] = float(values[i]) + offset
    return result

def apply_yaw_drift_correction(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Симметричная коррекция дрейфа XData (1-в-1 как во вьюере)."""
    df_out = df.copy()
    t_col = "Time" if "Time" in df.columns else "time"
    name_col = "Name" if "Name" in df.columns else None
    
    if not name_col or "XData" not in df.columns:
        return df_out, {"applied": False, "reason": "no Name/XData"}
        
    left_mask = df_out[name_col] == "ESP32_Sensor_1"
    right_mask = df_out[name_col] == "ESP32_Sensor_2"
    
    if not left_mask.any() or not right_mask.any():
        return df_out, {"applied": False, "reason": "missing foot sensor"}
        
    df_l = df_out[left_mask].sort_values(t_col)
    df_r = df_out[right_mask].sort_values(t_col)
    
    t_l_ms = pd.to_numeric(df_l[t_col], errors="coerce").to_numpy(dtype=float)
    t_r_ms = pd.to_numeric(df_r[t_col], errors="coerce").to_numpy(dtype=float)
    
    yaw_l = unwrap_angle_degrees(pd.to_numeric(df_l["XData"], errors="coerce").to_numpy(dtype=float))
    yaw_r = unwrap_angle_degrees(pd.to_numeric(df_r["XData"], errors="coerce").to_numpy(dtype=float))
    
    first = max(t_l_ms[0], t_r_ms[0])
    last = min(t_l_ms[-1], t_r_ms[-1])
    span_s = (last - first) / 1000.0
    if span_s < 31.0:
        return df_out, {"applied": False, "reason": f"span {span_s:.1f}s under 31s window"}
        
    shared_l = (t_l_ms >= first) & (t_l_ms <= last)
    t_shared_ms = t_l_ms[shared_l]
    yaw_shared_l = yaw_l[shared_l]
    
    t_r_u, idx_r = np.unique(t_r_ms, return_index=True)
    yaw_shared_r = np.interp(t_shared_ms, t_r_u, yaw_r[idx_r])
    divergence = yaw_shared_l - yaw_shared_r
    t_shared_s = t_shared_ms / 1000.0
    
    # 1-секундные блоки и 31-секундное сглаживание
    block_win_s = 1.0
    t0 = t_shared_s[0]
    num_blocks = int(np.floor((t_shared_s[-1] - t0) / block_win_s)) + 1
    block_idx = np.clip(np.floor((t_shared_s - t0) / block_win_s).astype(int), 0, num_blocks - 1)
    
    counts = np.bincount(block_idx, minlength=num_blocks)
    sums_t = np.bincount(block_idx, weights=t_shared_s, minlength=num_blocks)
    sums_v = np.bincount(block_idx, weights=divergence, minlength=num_blocks)
    
    valid = counts > 0
    block_t = sums_t[valid] / counts[valid]
    block_div = sums_v[valid] / counts[valid]
    
    win = max(3, int(round(31.0 / block_win_s)))
    if win % 2 == 0: win += 1
    
    s_div = pd.Series(block_div)
    smoothed = s_div.rolling(win, center=True, min_periods=1).median()
    smoothed = smoothed.rolling(win, center=True, min_periods=1).mean().to_numpy()
    
    # Ограничение скорости дрейфа (15 град/сек)
    curve = np.zeros_like(block_t)
    acc = 0.0
    for i in range(1, len(block_t)):
        ceil = 2.0 * 15.0 * (block_t[i] - block_t[i - 1])
        step = np.clip(smoothed[i] - smoothed[i - 1], -ceil, ceil)
        acc += step
        curve[i] = acc
        
    curve_t_ms = block_t * 1000.0
    corr_l = 0.5 * np.interp(t_l_ms, curve_t_ms, curve)
    corr_r = -0.5 * np.interp(t_r_ms, curve_t_ms, curve)
    
    df_out.loc[left_mask, "XData_Raw"] = df_out.loc[left_mask, "XData"]
    df_out.loc[right_mask, "XData_Raw"] = df_out.loc[right_mask, "XData"]
    
    df_out.loc[left_mask, "XData"] = yaw_l - corr_l
    df_out.loc[right_mask, "XData"] = yaw_r - corr_r
    
    info = {
        "applied": True,
        "span_s": span_s,
        "drift_deg": float(curve[-1] - curve[0]),
        "curve_t_s": block_t,
        "curve_deg": curve,
    }
    return df_out, info

df_corrected, drift_info = apply_yaw_drift_correction(df_raw)
print(f"Коррекция дрейфа: {drift_info['applied']} | Суммарный уход: {drift_info.get('drift_deg', 0):.1f}°")

# Визуализация дрейфа
plt.figure(figsize=(14, 4))
t_s_l = df_corrected[df_corrected["Name"] == "ESP32_Sensor_1"][t_col] / 1000.0
yaw_l_raw = unwrap_angle_degrees(df_corrected[df_corrected["Name"] == "ESP32_Sensor_1"]["XData_Raw"].values)
yaw_l_corr = df_corrected[df_corrected["Name"] == "ESP32_Sensor_1"]["XData"].values

plt.plot(t_s_l, yaw_l_raw, label="XData Unwrapped (Исходный с дрейфом)", color="gray", alpha=0.6)
plt.plot(t_s_l, yaw_l_corr, label="XData Corrected (С коррекцией дрейфа)", color="tab:blue", lw=1.5)
plt.title("Коррекция дрейфа гироскопа XData (ESP32_Sensor_1)")
plt.xlabel("Время (с)")
plt.ylabel("Угол Yaw (°)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Нейросетевая детекция контактов (WalkBiLSTM)

Запускаем модель `WalkBiLSTM` (`walk_gc_bilstm.pt`), которая находит моменты постановки и отрыва стопы для каждого шага.

In [ ]:
class _WalkBiLSTM(nn.Module):
    def __init__(self, n_features: int, hidden_size: int = 128, num_layers: int = 2, branch_channels: int = 48, proj_channels: int = 128, kernel_sizes = (3, 9, 21)):
        super().__init__()
        def branch(kernel_size: int):
            pad = kernel_size // 2
            return nn.Sequential(
                nn.Conv1d(n_features, branch_channels, kernel_size, padding=pad),
                nn.BatchNorm1d(branch_channels),
                nn.GELU(),
                nn.Conv1d(branch_channels, branch_channels, kernel_size, padding=pad),
                nn.BatchNorm1d(branch_channels),
                nn.GELU(),
            )
        self.branch_s = branch(kernel_sizes[0])
        self.branch_m = branch(kernel_sizes[1])
        self.branch_l = branch(kernel_sizes[2])
        fused = branch_channels * 3
        self.proj = nn.Sequential(nn.Conv1d(fused, proj_channels, 1), nn.BatchNorm1d(proj_channels), nn.GELU())
        self.lstm = nn.LSTM(proj_channels, hidden_size, num_layers=num_layers, batch_first=True, bidirectional=True)
        lstm_out = hidden_size * 2
        self.attn = nn.Sequential(nn.Linear(lstm_out, 64), nn.Tanh(), nn.Linear(64, 1))
        self.head = nn.Sequential(nn.Linear(lstm_out, 64), nn.GELU(), nn.Dropout(0.2), nn.Linear(64, 2))

    def forward(self, x):
        # x: (B, T, F)
        xb = x.transpose(1, 2)
        feat = torch.cat([self.branch_s(xb), self.branch_m(xb), self.branch_l(xb)], dim=1)
        feat = self.proj(feat).transpose(1, 2)
        out, _ = self.lstm(feat)
        logits = self.head(out)
        return logits

# Загрузка весов и скейлера
pt_path = MODEL_DIR / "walk_gc_bilstm.pt"
cfg_path = MODEL_DIR / "walk_gc_bilstm_config.pkl"
scl_path = MODEL_DIR / "walk_gc_bilstm_scaler.pkl"

with open(cfg_path, "rb") as f:
    config = joblib.load(f)
scaler = joblib.load(scl_path)

n_features = len(config["features"])
model = _WalkBiLSTM(n_features=n_features).to(device)
model.load_state_dict(torch.load(pt_path, map_location=device))
model.eval()
print(f"Модель WalkBiLSTM успешно загружена! Признаков: {n_features}, Window: {config.get('window', 100)}")

In [ ]:
def extract_walk_contacts(df: pd.DataFrame, model, config, scaler, weight_kg: float = 70.0) -> List[Dict[str, Any]]:
    t_col = "Time" if "Time" in df.columns else "time"
    name_col = "Name" if "Name" in df.columns else None
    all_contacts = []
    win = config.get("window", 100)
    step = config.get("stride", 50)
    bw_n = weight_kg * 9.80665

    for dev_name, foot in [("ESP32_Sensor_1", "left"), ("ESP32_Sensor_2", "right")]:
        sdf = df[df[name_col] == dev_name].sort_values(t_col).reset_index(drop=True)
        if sdf.empty: continue
        
        t_ms = pd.to_numeric(sdf[t_col], errors="coerce").to_numpy(dtype=float)
        t_s = t_ms / 1000.0
        acz = pd.to_numeric(sdf["AcZ"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        
        # Подготовка признаков
        feature_cols = config["features"]
        # Создание недостающих колонок при необходимости
        for col in feature_cols:
            if col not in sdf.columns:
                sdf[col] = 0.0
                
        X = sdf[feature_cols].to_numpy(dtype=np.float32)
        X_scaled = scaler.transform(X)
        
        N = len(X_scaled)
        probs_acc = np.zeros(N, dtype=np.float32)
        counts_acc = np.zeros(N, dtype=np.float32)
        
        # Оконный инференс
        with torch.no_grad():
            for i in range(0, N, step):
                end_i = min(i + win, N)
                chunk = X_scaled[i:end_i]
                if len(chunk) < win:
                    # Padding
                    pad_len = win - len(chunk)
                    chunk = np.pad(chunk, ((0, pad_len), (0, 0)), mode="edge")
                
                inp = torch.tensor(chunk, dtype=torch.float32).unsqueeze(0).to(device)
                out = model(inp) # (1, win, 2)
                prob = torch.softmax(out, dim=-1)[0, :end_i - i, 1].cpu().numpy()
                
                probs_acc[i:end_i] += prob
                counts_acc[i:end_i] += 1.0
                
        probs = probs_acc / np.maximum(counts_acc, 1.0)
        
        # Постпроцессинг (бисегментация)
        thresh = config.get("prob_threshold", 0.5)
        active = probs >= thresh
        diff = np.diff(np.concatenate(([0], active.astype(int), [0])))
        starts = np.where(diff == 1)[0]
        ends = np.where(diff == -1)[0]
        
        for s, e in zip(starts, ends):
            if e > s and e < len(t_s):
                dur_ms = t_ms[e - 1] - t_ms[s]
                dur_s = dur_ms / 1000.0
                if 0.08 <= dur_s <= 0.85:
                    pad = 0.1 * dur_s
                    mask = (t_s >= t_s[s] - pad) & (t_s <= t_s[e - 1] + pad)
                    pk_acz = float(np.max(np.abs(acz[mask]))) if np.any(mask) else float(np.max(np.abs(acz[s:e])))
                    
                    all_contacts.append({
                        "foot": foot,
                        "start_time_s": float(t_s[s]),
                        "end_time_s": float(t_s[e - 1]),
                        "duration_ms": round(float(dur_ms), 1),
                        "peak_force_n": round(float(weight_kg * (pk_acz / 10.0) * 9.80665), 1),
                        "peak_force_bw": round(float(pk_acz / 10.0), 2),
                        "confidence": round(float(np.mean(probs[s:e])), 3),
                    })

    all_contacts.sort(key=lambda c: c["start_time_s"])
    
    # Расчет Step Time и сил реакции опоры
    for i in range(len(all_contacts)):
        c = all_contacts[i]
        gct_s = c["duration_ms"] / 1000.0
        if i > 0:
            step_time_s = all_contacts[i]["start_time_s"] - all_contacts[i - 1]["start_time_s"]
            step_time_ms = step_time_s * 1000.0
            c["step_time_ms"] = round(step_time_ms, 1)
            
            # Физиологический диапазон шага: 0.12 - 0.70 сек
            if 0.12 <= step_time_s <= 0.70 and gct_s > 0:
                ratio = step_time_s / gct_s
                c["force_bw"] = round(ratio, 2)
                c["force_n"] = round(ratio * bw_n, 1)
            elif gct_s > 0:
                c["force_bw"] = c.get("peak_force_bw", 2.2)
                c["force_n"] = round(c["force_bw"] * bw_n, 1)
            else:
                c["force_bw"] = None
                c["force_n"] = None
        else:
            c["step_time_ms"] = None
            c["force_bw"] = c.get("peak_force_bw", 2.2)
            c["force_n"] = round(c["force_bw"] * bw_n, 1)
            
    return all_contacts

contacts = extract_walk_contacts(df_corrected, model, config, scaler, weight_kg=70.0)
n_l = sum(1 for c in contacts if c['foot'] == 'left')
n_r = sum(1 for c in contacts if c['foot'] == 'right')
print(f"Найдено шагов (WalkBiLSTM): {len(contacts)} (Left: {n_l}, Right: {n_r})")

## 4. Детекция 180° челночных разворотов (Yo-Yo Turns)

In [ ]:
class StandaloneTurnCalculator:
    def __init__(self, min_net_deg=35.0, win_ms=450.0, grid_ms=2.0, pad_ms=400.0, q_rise=0.08, min_amp_deg=100.0, merge_gap_ms=400.0, sg_window_length=277, sg_polyorder=2):
        self.min_net_deg = min_net_deg
        self.win_ms = win_ms
        self.grid_ms = grid_ms
        self.pad_ms = pad_ms
        self.q_rise = q_rise
        self.min_amp_deg = min_amp_deg
        self.merge_gap_ms = merge_gap_ms
        self.sg_window_length = sg_window_length
        self.sg_polyorder = sg_polyorder

    def _savgol_smooth(self, th):
        n = len(th)
        wl = min(self.sg_window_length, n)
        if wl % 2 == 0: wl -= 1
        if wl <= self.sg_polyorder: wl = self.sg_polyorder + 2
        if wl > n or wl < 3: return th
        return savgol_filter(th, window_length=wl, polyorder=self.sg_polyorder)

    def detect_turns(self, time_arr, angle_arr):
        t = pd.to_numeric(pd.Series(time_arr), errors="coerce").to_numpy(dtype=float)
        a = pd.to_numeric(pd.Series(angle_arr), errors="coerce").to_numpy(dtype=float)
        valid = np.isfinite(t) & np.isfinite(a)
        t, a = t[valid], a[valid]
        if len(t) < 10: return []
        
        order = np.argsort(t, kind="stable")
        t, a = t[order], a[order]
        t_u, idx = np.unique(t, return_index=True)
        theta = unwrap_angle_degrees(a)[idx]
        
        unit = 1.0 if np.max(t_u) > 3600 else 1e-3
        grid_step = self.grid_ms * unit
        tg = np.arange(t_u[0], t_u[-1], grid_step)
        if len(tg) < 10: return []
        
        th = np.interp(tg, t_u, theta)
        th = self._savgol_smooth(th)
        
        k = max(int(self.win_ms / self.grid_ms), 1)
        if k >= len(th): return []
        
        net = np.abs(th[k:] - th[:-k])
        mask = np.zeros(len(tg), dtype=bool)
        for i in np.flatnonzero(net >= self.min_net_deg):
            mask[i:i + k + 1] = True
            
        m = np.concatenate([[0], mask.astype(int), [0]])
        edges = np.flatnonzero(np.diff(m))
        regions = list(zip(edges[::2], edges[1::2] - 1))
        
        merged = []
        for s, e in regions:
            if merged and (tg[s] - tg[merged[-1][1]]) < (self.merge_gap_ms * unit):
                merged[-1] = (merged[-1][0], e)
            else:
                merged.append((s, e))
                
        out = []
        pad = int(self.pad_ms / self.grid_ms)
        for s, e in merged:
            lo_seg = th[max(s - pad, 0):s]
            hi_seg = th[e + 1:e + 1 + pad]
            if len(lo_seg) < 10 or len(hi_seg) < 10: continue
            lo, hi = float(np.median(lo_seg)), float(np.median(hi_seg))
            amp = hi - lo
            if abs(amp) < self.min_amp_deg: continue
            
            base = max(s - pad, 0)
            x = (th[base:e + 1 + pad] - lo) / amp
            mid = np.flatnonzero(x >= 0.5)
            if not len(mid): continue
            
            i = j = int(mid[0])
            while i > 0 and x[i] > self.q_rise: i -= 1
            while j < len(x) - 1 and x[j] < (1 - self.q_rise): j += 1
            
            t_start = float(tg[base + i])
            t_end = float(tg[base + j])
            out.append({
                "start_time_s": t_start / 1000.0,
                "end_time_s": t_end / 1000.0,
                "duration_ms": round(t_end - t_start, 1),
                "angle_deg": round(amp, 1),
                "direction": "left" if amp < 0 else "right",
            })
        return out

turn_detector = StandaloneTurnCalculator()
df_left_foot = df_corrected[df_corrected["Name"] == "ESP32_Sensor_1"].sort_values(t_col)
raw_turns = turn_detector.detect_turns(df_left_foot[t_col].values, df_left_foot["XData"].values)

for i, t in enumerate(raw_turns):
    t["turn_index"] = i

print(f"Детектировано челночных поворотов на 180°: {len(raw_turns)}")

## 5. Выделение шагов вокруг разворотов и расчет метрик

Для каждого из 85 разворотов извлекаются:
- **3 шага до разворота**: `step_-3`, `step_-2`, `step_-1`
- **Шаг разворота (Pivot foot)**: `turn_step`
- **1 шаг после разворота**: `step_+1`

In [ ]:
STEP_ROLES = ["step_-3", "step_-2", "step_-1", "turn_step", "step_+1"]
turn_records = []

for turn in raw_turns:
    t_start = turn["start_time_s"]
    t_end = turn["end_time_s"]
    
    pre_steps = [c for c in contacts if c["start_time_s"] < t_start]
    turn_steps = [c for c in contacts if (c["start_time_s"] <= t_end and c["end_time_s"] >= t_start)]
    post_steps = [c for c in contacts if c["start_time_s"] >= t_end]
    
    selected = {
        "step_-3": pre_steps[-3] if len(pre_steps) >= 3 else None,
        "step_-2": pre_steps[-2] if len(pre_steps) >= 2 else None,
        "step_-1": pre_steps[-1] if len(pre_steps) >= 1 else None,
        "turn_step": turn_steps[0] if len(turn_steps) >= 1 else None,
        "step_+1": post_steps[0] if len(post_steps) >= 1 else None,
    }
    
    record = {
        "turn_index": turn["turn_index"],
        "turn_start_s": round(turn["start_time_s"], 3),
        "turn_duration_ms": turn["duration_ms"],
        "turn_angle_deg": turn["angle_deg"],
    }
    
    for role in STEP_ROLES:
        st = selected[role]
        prefix = role
        if st is not None:
            record[f"{prefix}_foot"] = st["foot"]
            record[f"{prefix}_gct_ms"] = st["duration_ms"]
            record[f"{prefix}_step_time_ms"] = st["step_time_ms"]
            record[f"{prefix}_force_bw"] = st["force_bw"]
            record[f"{prefix}_force_n"] = st["force_n"]
        else:
            record[f"{prefix}_foot"] = None
            record[f"{prefix}_gct_ms"] = None
            record[f"{prefix}_step_time_ms"] = None
            record[f"{prefix}_force_bw"] = None
            record[f"{prefix}_force_n"] = None
            
    turn_records.append(record)

df_results = pd.DataFrame(turn_records)
print(f"Сформирована итоговая таблица: {df_results.shape[0]} разворотов")
df_results.head(10)

## 6. Сводная статистика и сравнение фаз шага

In [ ]:
stats_rows = []
role_labels = {
    "step_-3": "3-й шаг до (N-3)",
    "step_-2": "2-й шаг до (N-2)",
    "step_-1": "Предповоротный (N-1)",
    "turn_step": "Шаг разворота (Pivot)",
    "step_+1": "1-й шаг после (N+1)",
}

for role in STEP_ROLES:
    gct_m = df_results[f"{role}_gct_ms"].dropna().mean()
    gct_s = df_results[f"{role}_gct_ms"].dropna().std()
    st_m = df_results[f"{role}_step_time_ms"].dropna().mean()
    st_s = df_results[f"{role}_step_time_ms"].dropna().std()
    f_bw_m = df_results[f"{role}_force_bw"].dropna().mean()
    f_bw_s = df_results[f"{role}_force_bw"].dropna().std()
    f_n_m = df_results[f"{role}_force_n"].dropna().mean()
    
    stats_rows.append({
        "Фаза": role_labels[role],
        "GCT (мс)": f"{gct_m:.1f} ± {gct_s:.1f}",
        "Step Time (мс)": f"{st_m:.1f} ± {st_s:.1f}",
        "Force (BW)": f"{f_bw_m:.2f} ± {f_bw_s:.2f} BW",
        "Force (Ньютоны)": f"{f_n_m:.1f} Н",
    })

df_summary = pd.DataFrame(stats_rows)
df_summary

## 7. Графики распределения метрик

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

roles_disp = [role_labels[r] for r in STEP_ROLES]
gct_data = [df_results[f"{r}_gct_ms"].dropna().values for r in STEP_ROLES]
st_data = [df_results[f"{r}_step_time_ms"].dropna().values for r in STEP_ROLES]
f_data = [df_results[f"{r}_force_bw"].dropna().values for r in STEP_ROLES]

axes[0].boxplot(gct_data, labels=roles_disp, patch_artist=True, boxprops=dict(facecolor="lightblue"))
axes[0].set_title("Время контакта стопы (GCT, мс)")
axes[0].tick_params(axis="x", rotation=25)
axes[0].grid(True, alpha=0.3)

axes[1].boxplot(st_data, labels=roles_disp, patch_artist=True, boxprops=dict(facecolor="lightgreen"))
axes[1].set_title("Время шага (Step Time, мс)")
axes[1].tick_params(axis="x", rotation=25)
axes[1].grid(True, alpha=0.3)

axes[2].boxplot(f_data, labels=roles_disp, patch_artist=True, boxprops=dict(facecolor="salmon"))
axes[2].set_title("Сила реакции опоры (Force, BW)")
axes[2].tick_params(axis="x", rotation=25)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Сохранение результатов в CSV

In [ ]:
output_csv = "turn_steps_session_6546.csv"
df_results.to_csv(output_csv, index=False)
print(f"Результаты успешно сохранены в: {os.path.abspath(output_csv)}")